# Silver: ERP customer location
**Source:** `bronze.erp_loc_a101`  ->  **Target:** `silver.erp_customer_location`

**What this notebook does:**
- Remove extra spaces
- Fix customer ID (remove the `-` so it matches the CRM)
- Standardize country names (DE → Germany, US/USA → United States)
- Empty country → `n/a`
- Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.erp_loc_a101")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Fix customer ID
`AW-00011000` -> `AW00011000`.

In [0]:
df = df.withColumn("cid", F.regexp_replace("cid", "-", ""))

## 3. Standardize country names

In [0]:
df = df.withColumn(
    "cntry",
    F.when(F.col("cntry") == "DE", "Germany")
     .when(F.col("cntry").isin("US", "USA"), "United States")
     .when(F.col("cntry").isNull() | (F.col("cntry") == ""), "n/a")
     .otherwise(F.col("cntry"))
)

## 4. Rename columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customer_location")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.silver.erp_customer_location")
print("rows:", result.count())
result.groupBy("country").count().orderBy(F.desc("count")).display()